# VisDrone2019 MOT validation dataset inventory

Lists folders and files mounted by Kaggle under `/kaggle/input`, then writes inventory artifacts to `/kaggle/working`.

In [ ]:
import csv
import json
from pathlib import Path

input_root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd().parent / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

dataset_slug = "visdrone2019-mot-val"
dataset_root = input_root / dataset_slug
scan_root = dataset_root if dataset_root.exists() else input_root

directories = []
files = []

for path in sorted(scan_root.rglob("*")):
    rel_path = path.relative_to(scan_root).as_posix()
    if path.is_dir():
        directories.append(rel_path)
    elif path.is_file():
        files.append(
            {
                "path": rel_path,
                "size_bytes": path.stat().st_size,
                "suffix": path.suffix.lower(),
            }
        )

summary = {
    "input_root": str(input_root),
    "dataset_slug": dataset_slug,
    "dataset_root": str(dataset_root),
    "scan_root": str(scan_root),
    "dataset_root_exists": dataset_root.exists(),
    "directory_count": len(directories),
    "file_count": len(files),
    "total_size_bytes": sum(item["size_bytes"] for item in files),
    "top_level_entries": sorted(path.name for path in scan_root.iterdir()) if scan_root.exists() else [],
    "directories": directories,
    "files": files,
}

json_path = output_dir / "visdrone_inventory.json"
txt_path = output_dir / "visdrone_inventory.txt"
csv_path = output_dir / "visdrone_files.csv"

json_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["path", "size_bytes", "suffix"])
    writer.writeheader()
    writer.writerows(files)

lines = [
    f"input_root: {input_root}",
    f"dataset_root: {dataset_root}",
    f"dataset_root_exists: {dataset_root.exists()}",
    f"scan_root: {scan_root}",
    f"directory_count: {len(directories)}",
    f"file_count: {len(files)}",
    f"total_size_bytes: {summary['total_size_bytes']}",
    "",
    "Top-level entries:",
]
if summary["top_level_entries"]:
    lines.extend(f"* {entry}" for entry in summary["top_level_entries"])
else:
    lines.append("* none")
lines.extend(["", "First 100 directories:"])
lines.extend(f"* {path}" for path in directories[:100])
lines.extend(["", "First 200 files:"])
lines.extend(f"* {item['path']} ({item['size_bytes']} bytes)" for item in files[:200])
txt_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"Input root: {input_root}")
print(f"Dataset root: {dataset_root}")
print(f"Dataset root exists: {dataset_root.exists()}")
print(f"Scan root: {scan_root}")
print(f"Directories: {len(directories)}")
print(f"Files: {len(files)}")
print(f"Wrote: {json_path}")
print(f"Wrote: {txt_path}")
print(f"Wrote: {csv_path}")
